# BanglaSQL — Bangla Question → SQL

A question in Bangla about a university database → the SQL that answers it → the result.

| step | |
|---|---|
| 1 | Build the database |
| 2 | Build the dataset (241 templates → 2,170 pairs) |
| 3 | Tokenizer analysis |
| 3.5 | Training (off by default) |
| 4 | Load the model |
| 5 | Evaluate on the test set |
| 6 | Schema-constrained decoding |
| 7 | **Try it yourself** |

> Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

---
## Step 0 — Setup
Mount Drive, fetch the code, install dependencies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU — slower but works)')

In [ ]:
REPO_URL = 'https://github.com/mustafiz-07/BanglaSQL.git'

!rm -rf banglasql
!git clone -q -b model_contribution {REPO_URL} banglasql
%cd banglasql/
!pip install -q -r requirements_colab.txt
print('Code and dependencies ready.')

---
## Step 1 — The database

Synthetic university data, fixed seed so it is identical every run. The generator
guarantees every question has a non-empty answer and every filter excludes something,
so execution accuracy measures what it claims.

In [ ]:
!python create_database.py

import sqlite3, pandas as pd
con_show = sqlite3.connect('banglasql.db')
rows = []
q = "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
for (t,) in con_show.execute(q):
    n = con_show.execute('SELECT COUNT(*) FROM ' + t).fetchone()[0]
    cols = [r[1] for r in con_show.execute('PRAGMA table_info(' + t + ')')]
    rows.append({'table': t, 'rows': n, 'columns': ', '.join(cols)})
con_show.close()
pd.DataFrame(rows)

---
## Step 2 — The dataset

241 hand-written Bangla question/SQL templates → 2,170 pairs via three augmentations:

- **value-slot** — swap a literal in question and SQL
- **polarity** — flip বেশি/কম and আগে/পরে
- **paraphrase** — vary only the Bangla wording

Split is **template-level**: every variant stays with its base template, so nothing
derived from a training question reaches the test set.

In [ ]:
!python build_dataset.py

In [ ]:
import json, pandas as pd

stats = json.load(open('data/dataset_stats.json', encoding='utf-8'))
print('Base templates    :', stats['base_templates'])
print('After augmentation:', stats['after_augmentation_dedup']['total'], 'pairs')
print()
split_tbl = pd.DataFrame({
    'pairs': stats['splits'],
    'base templates': stats['templates_per_split'],
    'unique SQL': stats['unique_sql_per_split'],
})
display(split_tbl)
print('Leakage check —', stats['sql_leakage'])
print('Query types:', stats['query_types']['total'],
      '| unseen in train:', stats['query_types']['test_unseen_in_train'], '(empty = good)')

In [ ]:
# A few training pairs, so the input/output format is concrete.
train = json.load(open('data/dataset_train.json', encoding='utf-8'))
for ex in train[:3]:
    print('Q   ', ex['bangla_question'])
    print('SQL ', ex['sql_query'])
    print()

---
## Step 3 — Tokenizer analysis

BanglaT5 covers Bangla far better than mT5 with an eighth of the vocabulary.
Also sizes sequence lengths from the longest target, so no gold query is truncated.

In [ ]:
!python preprocess_check.py

---
## Step 3.5 — Training

**Off by default** — 45–70 min on a T4. Set `RUN_TRAINING = True` to reproduce it.

LR 2e-4 · batch 8 · 25 epochs, early-stop patience 10 · fp16 off (T5 overflows) ·
best checkpoint chosen on dev **execution accuracy**.

Output: `checkpoints/best_model`.

In [ ]:
import subprocess, sys

RUN_TRAINING = False   # set True to train from scratch (~45-70 min on a T4)

if RUN_TRAINING:
    subprocess.run([sys.executable, 'train.py'], check=True)
else:
    print('Skipped. Step 4 loads the checkpoint that training already produced.')
    print('Set RUN_TRAINING = True above to reproduce training end to end.')

In [ ]:
# Training curves, if a run has produced logs/train_history.json.
import os, json

if os.path.exists('logs/train_history.json'):
    import matplotlib.pyplot as plt
    history = json.load(open('logs/train_history.json', encoding='utf-8'))
    train_pts = [(h['epoch'], h['loss']) for h in history if 'loss' in h]
    eval_pts = [(h['epoch'], h['eval_loss']) for h in history if 'eval_loss' in h]
    acc_pts = [(h['epoch'], h['eval_execution_accuracy'])
               for h in history if 'eval_execution_accuracy' in h]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    if train_pts:
        ax1.plot(*zip(*train_pts), label='train loss')
    if eval_pts:
        ax1.plot(*zip(*eval_pts), label='dev loss')
    ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(); ax1.set_title('Loss')
    if acc_pts:
        ax2.plot(*zip(*acc_pts), color='green')
    ax2.set_xlabel('epoch'); ax2.set_ylabel('execution accuracy')
    ax2.set_title('Dev execution accuracy')
    plt.tight_layout(); plt.show()
else:
    print('No logs/train_history.json — run training above to produce the curves.')

---
## Step 4 — The model

**`csebuetnlp/banglat5`** — T5-style encoder–decoder, ~297M parameters, pretrained on Bangla.

Loads the checkpoint that training produced.

In [ ]:
import os
CKPT = '/content/drive/MyDrive/BanglaSQL/checkpoints/bestmodel'

if not os.path.isdir(CKPT):
    parent = os.path.dirname(CKPT)
    print('Not found:', CKPT)
    print('Contents of', parent, ':', os.listdir(parent) if os.path.isdir(parent) else 'missing')
    raise SystemExit('Edit CKPT above to point at the folder holding config.json and model.safetensors.')

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from common import load_config

tokenizer = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForSeq2SeqLM.from_pretrained(CKPT)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
config = load_config(CKPT)

print('Checkpoint :', CKPT)
print('Parameters :', format(sum(p.numel() for p in model.parameters()), ','))
print('Max in/out :', config['max_input_length'], '/', config['max_target_length'])
print('Input      : question' + (' + schema' if config['include_schema'] else ' only'))

---
## Step 5 — Evaluation

**Execution accuracy** is the primary metric: run the predicted and gold queries, compare
result sets. It accepts any SQL that returns the right answer.

4 beams + **execution-guided decoding** (Wang et al., 2018) — the highest-scoring beam that runs.

In [ ]:
!python evaluate.py --split test --model "{CKPT}" --tag demo

In [ ]:
import json, pandas as pd

res = json.load(open('logs/test_demo_results.json', encoding='utf-8'))
m, t1 = res['metrics'], res['top1_metrics']

summary = pd.DataFrame({
    'top-1 beam': [t1['execution_accuracy'], t1['exact_match'], t1['validity_rate']],
    'execution-guided': [m['execution_accuracy'], m['exact_match'], m['validity_rate']],
}, index=['execution accuracy', 'exact match', 'validity rate'])
display(summary.map(lambda v: format(100 * v, '.1f') + '%'))

gain = 100 * (m['execution_accuracy'] - t1['execution_accuracy'])
print('Execution-guided decoding is worth', format(gain, '+.1f'), 'points.')

In [ ]:
# Per-clause accuracy — the most readable diagnostic signal.
comp = pd.DataFrame([{'component': k, 'accuracy': format(100 * v['accuracy'], '.1f') + '%'}
                     for k, v in sorted(res['component_accuracy'].items())])
fails = pd.DataFrame(sorted(res['failure_categories'].items(), key=lambda kv: -kv[1]),
                     columns=['failure category', 'count'])
display(comp, fails)

In [ ]:
# Accuracy by question type — the spread matters more than the average.
qt = pd.DataFrame([{'query type': k, 'n': v['n'], 'accuracy': 100 * v['execution_accuracy']}
                   for k, v in res['by_query_type'].items()]).sort_values('accuracy')
ax = qt.plot.barh(x='query type', y='accuracy', figsize=(8, 7), legend=False,
                  title='Execution accuracy by query type (%)')
ax.set_xlabel('execution accuracy (%)')
ax.figure.tight_layout()

---
## Step 6 — Schema-constrained decoding

Execution-guided decoding selects among finished beams, so it cannot help when *every*
beam is invalid. This constrains generation instead: at each step the vocabulary is masked
so only schema-valid continuations remain.

| constraint | rule |
|---|---|
| table position | after `FROM`/`JOIN`, only a real table name |
| qualified column | after `c.`, only a column of `courses` |
| **EOS gating** | the query may not end while an alias is unbound |

EOS gating handles SQL's forward reference: `SELECT c.course_name` comes *before*
`FROM courses c`, so the decoder records an obligation and refuses to stop until
`courses` is joined.

Applied **only when no beam executes**. Validity 96.0% → 98.2%, malformed queries 13 → 6.

The cells below verify the grammar admits every gold query, then show the masking.

In [ ]:
# Verify the grammar admits every gold query before trusting it to constrain anything.
# This gate caught two bugs that would otherwise have looked like a mysterious accuracy drop.
from schema_grammar import SchemaIndex, TokenFilter, parse_prefix, FREE
from constrained_decode import ConstrainedDecoder

decoder = ConstrainedDecoder(tokenizer, max_length=int(config['max_target_length']))
print('Schema indexed:', len(decoder.schema.table_names), 'tables,',
      len(decoder.schema.all_columns), 'columns')

templates = json.load(open('data/templates.json', encoding='utf-8'))
blocked = 0
for tpl in templates:
    ids = tokenizer(tpl['sql_query'], add_special_tokens=False)['input_ids']
    for i in range(1, len(ids)):
        if ids[i] not in decoder.allowed_tokens(ids[:i]):
            blocked += 1
            break
print('Gold queries the grammar would block:', blocked, 'of', len(templates))

In [ ]:
# Demonstrate the constraint on a single step.
probe = 'SELECT * FROM '
state = parse_prefix(probe, decoder.schema)
allowed = decoder.filter.allowed(state.names, state.partial, state.needs_space)
print('After', repr(probe), '->', len(allowed), 'of',
      len(decoder.filter.all_ids), 'tokens remain reachable')
print('They spell:', sorted(decoder.schema.table_names))
print()

pending = parse_prefix('SELECT c.course_name FROM students s', decoder.schema).pending
print('After "SELECT c.course_name FROM students s" the alias', sorted(pending),
      'is still unbound -> end-of-sequence is refused.')

---
# Step 7 — Try it yourself

Type a Bangla question or pick one from the dropdown, then press **Run**.

Tick **show all candidate queries** to see the 4 beams and which one was chosen.

In [ ]:
import pandas as pd, torch
from common import format_input, open_readonly_db, pick_executable

con = open_readonly_db()
device = 'cuda' if torch.cuda.is_available() else 'cpu'


def generate_sql(question, num_beams=4, constrained=False):
    """Bangla question -> (chosen SQL, all candidate beams in model-score order)."""
    enc = tokenizer(format_input(question, config),
                    max_length=int(config['max_input_length']),
                    truncation=True, return_tensors='pt').to(device)
    kwargs = dict(max_length=int(config['max_target_length']),
                  num_beams=num_beams, num_return_sequences=num_beams,
                  early_stopping=True)
    if constrained:
        kwargs['prefix_allowed_tokens_fn'] = decoder.prefix_fn()
    with torch.no_grad():
        out = model.generate(**enc, **kwargs)
    beams = tokenizer.batch_decode(out, skip_special_tokens=True)
    return pick_executable(beams, con), beams


def run_and_show(sql, limit=50):
    """Execute read-only and return (dataframe, error message)."""
    try:
        cur = con.execute(sql)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchmany(limit), columns=cols), None
    except Exception as exc:
        return None, str(exc)


print('Ready. Device:', device)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

EXAMPLES = [
    'সকল শিক্ষার্থীর তালিকা দাও।',
    'যেসব শিক্ষার্থীর CGPA ৩.৫-এর বেশি তাদের নাম দাও।',
    'প্রতিটি বিভাগে কতজন শিক্ষার্থী আছে?',
    'সবচেয়ে বেশি CGPA কত?',
    'গড় CGPA কত?',
    'প্রতিটি কোর্সের নাম ও ক্রেডিট দেখাও।',
    'কোন কোর্সে সবচেয়ে বেশি শিক্ষার্থী নথিভুক্ত?',
    'CGPA অনুযায়ী অবরোহী ক্রমে ১০ জন শিক্ষার্থীর নাম দাও।',
]

picker = widgets.Dropdown(options=['(type your own below)'] + EXAMPLES,
                          description='Example:',
                          layout=widgets.Layout(width='90%'),
                          style={'description_width': '90px'})
box = widgets.Text(value=EXAMPLES[1], description='Question:',
                   placeholder='Write a question in Bangla...',
                   layout=widgets.Layout(width='90%'),
                   style={'description_width': '90px'})
constrain = widgets.Checkbox(value=False, description='schema-constrained decoding')
show_beams = widgets.Checkbox(value=False, description='show all candidate queries')
button = widgets.Button(description='Run', button_style='primary', icon='play')
out = widgets.Output()


def _on_pick(change):
    if change['new'] in EXAMPLES:
        box.value = change['new']


def _on_run(_):
    question = box.value.strip()
    with out:
        clear_output()
        if not question:
            print('Please enter a question.')
            return
        display(Markdown('**Question:** ' + question))
        sql, beams = generate_sql(question, constrained=constrain.value)
        display(Markdown('**Generated SQL:**'))
        display(Markdown('```sql\n' + sql + '\n```'))
        df, err = run_and_show(sql)
        if err:
            display(Markdown('**Did not execute:** `' + err + '`'))
        elif df.empty:
            display(Markdown('Executed successfully, but returned no rows.'))
        else:
            display(Markdown('**Result** (' + str(len(df)) + ' rows shown):'))
            display(df)
        if show_beams.value:
            display(Markdown('**All candidates, in model-score order:**'))
            for i, b in enumerate(beams):
                status = 'runs ' if run_and_show(b)[1] is None else 'fails'
                mark = '   <-- chosen' if b == sql else ''
                print(str(i + 1) + '. [' + status + '] ' + b + mark)


picker.observe(_on_pick, names='value')
button.on_click(_on_run)
display(widgets.VBox([picker, box, widgets.HBox([constrain, show_beams]), button, out]))

### If the widget does not render
Colab sometimes blocks widgets. Same thing as a plain function call — edit and re-run.

In [ ]:
def ask(question, constrained=False, show_beams=False):
    print('Q:  ', question)
    sql, beams = generate_sql(question, constrained=constrained)
    print('SQL:', sql)
    df, err = run_and_show(sql)
    if err:
        print('Did not execute:', err)
    elif df.empty:
        print('Executed, no rows returned.')
    else:
        display(df)
    if show_beams:
        print()
        for i, b in enumerate(beams):
            print(' ', i + 1, b)
    return sql


ask('যেসব শিক্ষার্থীর CGPA ৩.৫-এর বেশি তাদের নাম দাও।')

---
## Summary

| component | |
|---|---|
| **Database** | 6 tables, 5,638 rows, fixed seed |
| **Dataset** | 241 templates over 28 query types → 2,170 pairs, template-level split |
| **Model** | BanglaT5 (~297M) fine-tuned for Bangla → SQL |
| **Decoding** | 4 beams, execution-guided selection, schema-constrained fallback |
| **Interface** | this notebook, and `app.py` (Streamlit) |

**Test set:** 59.6% execution accuracy · 51.1% exact match · 98.2% validity.

```bash
BANGLASQL_MODEL="path/to/checkpoint" streamlit run app.py
```